# Récupérer les données géographiques (pays, villes)



In [1]:
#chargement des modules
import pandas as pd
import datetime
import os
import json
from io import StringIO
import numpy as np
import plotly.express as px
import re

In [2]:
import s3fs #pour connecter au bucket

In [3]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [4]:
file_path = BUCKET_OUT + "/" + "Data_bso/fichier_from_mesr/bso-publications-latest_199318270_enriched.jsonl"
print(file_path)
list_id_pub = []
datas =[]
with fs.open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        try:
            data = json.loads(line)
            # Extract specific fields
            ids = data.get('all_ids')
            list_id_pub.append(ids[0])
            datas.append(data)
            
        except json.JSONDecodeError:
            continue  # Skip invalid lines

aluneau/Data_bso/fichier_from_mesr/bso-publications-latest_199318270_enriched.jsonl


La cellule ci-dessous récupèrent les données qui nous intéressent depuis le fichier Json. Ces données seront ensuitre structurées dans un dataframe constitué d'une ligne par publication. Certaines de ces données ne sont en effet pas disponibles dans le fichier "bso-publications-latest_199318270_enriched.csv".


In [5]:
authors_halid_s = [] #identifiant HAl au format string : prenom.nom
authors_halid_i = [] # identifiant hal au format numerique, ex: 191919
authors_fullname = [] #nom prénom de l'auteur
authors_orcid = [] #orcid de l'auteur
structures_id = [] #hal id de la structure
structures_country = [] #pays de la structurre (si plusieurs structures, on prend la première dans l'ordre d'apparition)
structures_address = [] #addresse de la structurre (si plusieurs structures, on prend la première dans l'ordre d'apparition)
hal_class_code = [] #Classification hal
bso_class = [] #Classification disciplinaire du BSO
pub_year = [] #Année de publication
pub_date = [] #date de publication
pub_hal_id = [] #hal id de la publication
is_oa = [] #statut d'ouverture en 2024
list_ids = [] #id de la publication dans le fichier du BSO
list_rors = []

for n, x in enumerate(list_id_pub):
    
    dict_pub = datas[n]
    if 'authors' in dict_pub:
        #print(n, x)
        list_author = []
        list_author_id_s = []
        list_author_id_i = []
        list_author_orcid = []
        list_structures_id = []
        list_country = []
        list_address = []
        list_keyword = []
        list_ids.append(x)
        
        for a in dict_pub['authors']:
            #print(a)
            
            if  'full_name' in a:
                list_author.append(a['full_name'])
            else:
                list_author.append('no full_name')
            if 'id_hal_s' in a:
                list_author_id_s.append(a['id_hal_s'])
            else:
                list_author_id_s.append('no_id_hal_s')
            if 'id_hal_i'  in a:
                list_author_id_i.append(a['id_hal_i'])
            else:
                list_author_id_i.append('no_id_hal_i')
            if 'orcid' in a:
                list_author_orcid.append(a['orcid'])
            else:
                list_author_orcid.append('no orcid')
            if "affiliations" in a and len(a["affiliations"]) > 0 :
                dict_affil = a["affiliations"][0]
                if "country" in dict_affil:
                    list_country.append(dict_affil["country"])
                else:
                    list_country.append("no country")
                if "name" in dict_affil:
                    list_address.append(dict_affil["name"])
                else:
                    list_address.append("no address")
            else:
                list_country.append("no country")
                list_address.append("no address")
    
        authors_halid_s.append("|".join(list_author_id_s))
        authors_halid_i.append("|".join(list_author_id_i))
        authors_fullname.append("|".join(list_author))
        authors_orcid.append("|".join(list_author_orcid))
        structures_country.append("|".join(list_country))
        structures_address.append("|".join(list_address))

        if 'hal' in x:
            pub_hal_id.append(x[3:])
        else:
            pub_hal_id.append(np.nan)
        if 'hal_classification' in dict_pub:
            hal_classification = [h['code'] for h in dict_pub['hal_classification']]
            hal_class_code.append('|'.join(hal_classification))
        else:
            hal_class_code.append('no hal_classification')
        bso_class.append(dict_pub['bso_classification'])
        if 'keywords' in dict_pub:
            for kw in dict_pub["keywords"]:
                list_keyword.append(kw["keyword"])
        else:
            list_keyword.append('no keywords')
        affiliations_i = [inst for inst in dict_pub['bso_local_affiliations']  if re.search(r'[a-z]', inst) is None ]
        structures_id.append('|'.join(affiliations_i))
        #pub_date.append(dict_pub['published_date'])
        pub_year.append(dict_pub['year'])
        rors = dict_pub['rors']
        list_rors.append("|".join(rors))

        for y in dict_pub['oa_details']:
            if y == "2024Q4":
                dict_oa = dict_pub['oa_details'][y]
                #published_date.append(dict_pub['external_ids']['published_date'])
                is_oa.append(dict_oa["is_oa"])
    else:
        pass
        
    

In [6]:
data_columns = {
    "id": list_ids,
    'hal_id': pub_hal_id,
    "pub_year" : pub_year,
    'authors' : authors_fullname,
    'authhalid_s': authors_halid_s,
    'authhalid_i':authors_halid_i,
    'orcid': authors_orcid,
    'struchal_id': structures_id,
    'struc_country': structures_country,
    'struc_address': structures_address,
    'rors': list_rors,
    'hal_classif' : hal_class_code,
    'bso_classif' : bso_class,
    'is_oa': is_oa
}

df0 = pd.DataFrame(data= data_columns)
df0

,id,hal_id,pub_year,authors,authhalid_s,authhalid_i,orcid,struchal_id,struc_country,struc_address,rors,hal_classif,bso_classif,is_oa
0,halhal-03933378,hal-03933378,2019,Jérôme Livet|François Capron|Philippe Blanchar...,jerome-livet|francois-capron|philippe-blanchar...,176081|15426|18849|245|6836|2483|no_id_hal_i|1...,0000-0002-1785-776X|no orcid|0000-0003-0829-77...,7550|199318270|205557|301475|1059646|11141|178...,France|France|France|France|France|France|Fran...,Institut national de recherches archéologiques...,https://ror.org/04wez5e68|https://ror.org/02ww...,shs.archeo_,Humanities,False
1,halhal-03895949,hal-03895949,2017,Katerina Chryssanthaki-Nagle,no_id_hal_s,no_id_hal_i,no orcid,7550|199318270|11141|300125|1003|441569|105964...,no country,"Archéologies et Sciences de l'Antiquité, UMR70...",https://ror.org/04wez5e68,shs.archeo_|shs.art_|shs.class_|shs.hist_|shs....,Humanities,False
2,halhal-01789653,hal-01789653,2017,Makis Solomos,no_id_hal_s,no_id_hal_i,no orcid,199318270|11141|188359,no country,"Esthétique, musicologie, danse et création mus...",https://ror.org/04wez5e68,shs.musiq_,Humanities,True
3,halhal-03807832,hal-03807832,2014,Isabelle Launay,no_id_hal_s,no_id_hal_i,no orcid,199318270|11141|188359,no country,"Esthétique, musicologie, danse et création mus...",https://ror.org/04wez5e68,shs_|shs.musiq_,Humanities,False
4,doi10.3917/cliop.028.0029,NaN,2022,Léandro de Lajonquière|Floriana Baldassi D’Arrigo,no_id_hal_s|no_id_hal_s,no_id_hal_i|no_id_hal_i,no orcid|no orcid,199318270|199411117|420786|106111|11141,no country|no country,no address|no address,https://ror.org/04wez5e68|https://ror.org/05gg...,shs.edu_|shs.psy_,Social sciences,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48081,halhal-04095338,hal-04095338,2020,Céline Clément,celine-clement-nanterre,1253116,no orcid,439352|441569|199318270|116205|443665|11141,no country,"Genre, Travail, Mobilités, Cresppa-GTM, CRESPP...",https://ror.org/04wez5e68,shs.demo_|shs.socio_,Social sciences,False
48082,halhal-03299923,hal-03299923,2020,Martine Créac'H,martine-creach,746578,no orcid,199318270|1007102|11141,no country,"Fabrique du littéraire, EA7322, Fablitt, UP8, ...",https://ror.org/04wez5e68,shs_|shs.litt_,Humanities,False
48083,halhal-04366816,hal-04366816,2023,Brigitte Bardin|Anne Burguet|Sophie Henry|Nico...,brigitte-bardin|annette-burguet|sophie-henry|n...,1316897|753990|1327258|1268072,no orcid|0009-0005-8979-2706|no orcid|0000-000...,199318270|193113842|217752|1052984|11141|42812...,France|France|France|France,"Laboratoire Parisien de Psychologie Sociale, E...",https://ror.org/04wez5e68|https://ror.org/02v6...,scco.psyc_,Humanities,True
48084,halhal-02120667,hal-02120667,2018,Valentina M. Azzarà|A.P. de Rorre|Jean-Francoi...,no_id_hal_s|no_id_hal_s|jean-francois-berger|g...,no_id_hal_i|no_id_hal_i|742611|13749,no orcid|no orcid|0000-0002-4360-7262|0000-000...,7550|199318270|301232|11141|300311|196901920|1...,France|no country|France|France,"Archéologies et Sciences de l'Antiquité, UMR70...",https://ror.org/04wez5e68|https://ror.org/050j...,shs.archeo_|sde.es_|sdu.stu.gp_,"Earth, Ecology, \nEnergy and applied biology",False


In [13]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_publi_from_bso_with_address.csv", "w") as file_out:
    df0.to_csv(file_out, sep=",", index= False)

In [11]:
file_path = BUCKET_OUT + "/" + "Data_bso/fichier_from_mesr/bso-publications-latest_199318270_enriched.csv"
with fs.open(file_path) as csvin:
    df1 = pd.read_csv(csvin, sep =";")

df1.columns

Index(['observation_date', 'id', 'doi', 'pmid', 'hal_id', 'year', 'title',
       'journal_issns', 'journal_issn_l', 'journal_name', 'publisher',
       'publisher_dissemination', 'bso_classification', 'lang', 'genre',
       'amount_apc_EUR', 'apc_source', 'domains', 'detected_countries',
       'bso_local_affiliations', 'rors', 'funding_anr', 'funding_europe',
       'bsso_classification', 'is_oa', 'is_oa_hal', 'oa_host_type',
       'journal_is_in_doaj', 'journal_is_oa', 'unpaywall_oa_status',
       'oa_colors', 'licence_publisher', 'licence_repositories',
       'repositories', 'software_mentions', 'data_mentions', 'software_used',
       'software_created', 'software_shared', 'data_used', 'data_created',
       'data_shared', 'missing_doi_in_hal', 'has_doi_in_hal', 'doi_in_hal',
       'bso_country'],
      dtype='object')

## Compléter les adresses manquante à partir des affiliations retrouvées via Crossref

## Retrouver les informations géographiques à partir du ROR

L'objectif est d'utiliser l'api de ROR pour récupérer des données géographiques à partir de l'identifiation ROR des institutions. Le problème est que toutes les institutions recencées dans le BSO n'ont pas de ROR. L'analyse des données montre qu'on dispose du ROR uniquement pour les établissement français.

On utilise la librairie requests pour interroger l'API

In [44]:
df01 = df0[["id","rors"]]
df01["rors"] = df01.rors.str.split("|")
df_ror = df01.explode("rors")


/tmp/ipykernel_1540/4143035616.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df01["rors"] = df01.rors.str.split("|")


In [45]:
import requests
from requests.exceptions import HTTPError
import urllib.parse
import tqdm

In [46]:
ROR_API_ENDPOINT = "https://api.dev.ror.org/organizations"


params = {"query":"1m84wm78"} # le nombre de ligne. Par défaut, l'api affiche les 30 première ligne et le maximum fixé est de 10000.

In [47]:
query_response = requests.get(ROR_API_ENDPOINT + '/' + "01m84wm78").json()

On affiche la structure des données

In [48]:
query_response

{'admin': {'created': {'date': '2018-11-14', 'schema_version': '1.0'},
  'last_modified': {'date': '2026-01-29', 'schema_version': '2.1'}},
 'domains': [],
 'established': 1969,
 'external_ids': [{'all': ['grid.11619.3e'],
   'preferred': 'grid.11619.3e',
   'type': 'grid'},
  {'all': ['0000 0001 2152 2279'], 'preferred': None, 'type': 'isni'},
  {'all': ['Q459026'], 'preferred': None, 'type': 'wikidata'}],
 'id': 'https://ror.org/01m84wm78',
 'links': [{'type': 'website', 'value': 'http://www.univ-rennes2.fr/'},
  {'type': 'wikipedia',
   'value': 'http://en.wikipedia.org/wiki/University_of_Rennes_2_%E2%80%93_Upper_Brittany'}],
 'locations': [{'geonames_details': {'continent_code': 'EU',
    'continent_name': 'Europe',
    'country_code': 'FR',
    'country_name': 'France',
    'country_subdivision_code': 'BRE',
    'country_subdivision_name': 'Brittany',
    'lat': 48.11198,
    'lng': -1.67429,
    'name': 'Rennes'},
   'geonames_id': 2983990}],
 'names': [{'lang': 'en',
   'types':

In [49]:
for n, x in enumerate(query_response["names"]):
    if 'ror_display' in x["types"]:
        print(x)

{'lang': 'fr', 'types': ['label', 'ror_display'], 'value': 'Université Rennes 2'}


Nous voulons récupérer les informations contenues dans "locations" :

```
locations': [{'geonames_details': {'continent_code': 'EU',
    'continent_name': 'Europe',
    'country_code': 'FR',
    'country_name': 'France',
    'country_subdivision_code': 'BRE',
    'country_subdivision_name': 'Brittany',
    'lat': 48.11198,
    'lng': -1.67429,
    'name': 'Rennes'},

```

In [50]:
dict_ror = {}
list_row = []
for ror in tqdm.tqdm(df_ror.rors.unique(), total = df_ror.rors.nunique()):
    dict_row ={}
    rorid = ror.split("/")[-1]
    #print(rorid)
    dict_ror[ror] = rorid
    dict_row["ror"] = ror
    data = requests.get(ROR_API_ENDPOINT + '/' + rorid).json()

    for n, x in enumerate(data["names"]):
        if 'ror_display' in x["types"]:
            dict_row["nom"] = x["value"]
    locations_data = data["locations"][0]["geonames_details"]
    for i in locations_data:
        dict_row[i] = locations_data[i]
        if i == "name":
            dict_row["city"] = locations_data[i]
    dict_row["geonames_id"] = data["locations"][0]["geonames_id"]
    list_row.append(dict_row)


        
    
    
    

100%|██████████| 155/155 [00:21<00:00,  7.23it/s]


In [51]:
df_inst = pd.DataFrame.from_dict(list_row)

In [52]:
df_inst.country_name.value_counts()
df_inst

,ror,nom,continent_code,continent_name,country_code,country_name,country_subdivision_code,country_subdivision_name,lat,lng,name,city,geonames_id
0,https://ror.org/04wez5e68,Université Paris 8,EU,Europe,FR,France,IDF,Île-de-France,48.85341,2.34880,Paris,Paris,2988507
1,https://ror.org/02wwzvj46,Université de Tours,EU,Europe,FR,France,CVL,Centre-Val de Loire,47.39484,0.70398,Tours,Tours,2972191
2,https://ror.org/057qpr032,Université de Bordeaux,EU,Europe,FR,France,NAQ,Nouvelle-Aquitaine,44.84124,-0.58046,Bordeaux,Bordeaux,3031582
3,https://ror.org/05ggc9x40,Université Paris-Est Créteil,EU,Europe,FR,France,IDF,Île-de-France,48.79266,2.46569,Créteil,Créteil,3022530
4,https://ror.org/01qhqcj41,Université de Technologie de Troyes,EU,Europe,FR,France,GES,Grand Est,48.30073,4.08524,Troyes,Troyes,2971549
...,...,...,...,...,...,...,...,...,...,...,...,...,...
150,https://ror.org/03pnp1a74,Lamsade,EU,Europe,FR,France,IDF,Île-de-France,48.85341,2.34880,Paris,Paris,2988507
151,https://ror.org/042xmz674,Laboratoire de Physique et Chimie des Nano-Objets,EU,Europe,FR,France,OCC,Occitanie,43.60426,1.44367,Toulouse,Toulouse,2972315
152,https://ror.org/04fm0sh33,Laboratoire de Planétologie et Géosciences,EU,Europe,FR,France,PDL,Pays de la Loire,47.21725,-1.55336,Nantes,Nantes,2990969
153,https://ror.org/022zdgq74,École Nationale de l’Aviation Civile,EU,Europe,FR,France,OCC,Occitanie,43.60426,1.44367,Toulouse,Toulouse,2972315


In [53]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_geographical_info_french_affiliation.csv", "w") as file_out:
    df_inst.to_csv(file_out, sep=",", index= False)

## Retrouver le ror à partir du nom des universités

Dans les données enrichies, le ror est présent uniquement pour les établissement français. On va essayer de récupérer le ror à partir de l'adresse des structures (lorsqu'elle est indiquée)

In [82]:
file_path = BUCKET_OUT + "/" + "Data_bso/outputs/enriched_data/geolocation/2026-03-11_affil_crossref.csv"
with fs.open(file_path) as csvin:
    dfcr = pd.read_csv(csvin, sep =",")


dfcr1 = dfcr.loc[dfcr.affiliation!="no affil"]

df01 = df0.merge(dfcr1[["id","affiliation"]], on = ["id"], how = "left").merge(df1[["id","title"]], on = ["id"], how = "left")#.reset_index()


In [99]:
df02 = df01[["id","struc_address","affiliation","title"]]
df02["address"] = df02.struc_address.str.split("|")
df_ad = df02.explode("address")
df_ad2 = df_ad.loc[~(df_ad.address.str.lower().str.contains('france')) & ~(df_ad.affiliation.isna())].reset_index().drop(columns=["index"])
df_ad2


/tmp/ipykernel_1540/1215835960.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df02["address"] = df02.struc_address.str.split("|")


,id,struc_address,affiliation,title,address
0,doi10.1590/198053142861,Nos últimos trinta anos Nos últimos trinta anos,Universidade Paris 13; International Associati...,A educação infantil e a questão da escola: o c...,Nos últimos trinta anos Nos últimos trinta anos
1,doi10.1080/15592324.2021.2004769,no address|no address|no address|no address|no...,Laboratoire Interdisciplinaire Des Énergies de...,Our sisters the plants? notes from phylogeneti...,no address
2,doi10.1080/15592324.2021.2004769,no address|no address|no address|no address|no...,Laboratoire Interdisciplinaire Des Énergies de...,Our sisters the plants? notes from phylogeneti...,no address
3,doi10.1080/15592324.2021.2004769,no address|no address|no address|no address|no...,Laboratoire Interdisciplinaire Des Énergies de...,Our sisters the plants? notes from phylogeneti...,no address
4,doi10.1080/15592324.2021.2004769,no address|no address|no address|no address|no...,Laboratoire Interdisciplinaire Des Énergies de...,Our sisters the plants? notes from phylogeneti...,no address
...,...,...,...,...,...
57630,doi10.1080/09537287.2022.2037166,no address|no address|no address,"QUARTZ Laboratory, IUT de Montreuil, Montreuil...",A comparative study of overall equipment effec...,no address
57631,doi10.1080/09537287.2022.2037166,no address|no address|no address,"QUARTZ Laboratory, IUT de Montreuil, Montreuil...",A comparative study of overall equipment effec...,no address
57632,doi10.1080/09537287.2022.2037166,no address|no address|no address,"QUARTZ Laboratory, Supmeca, Saint-Ouen, France",A comparative study of overall equipment effec...,no address
57633,doi10.1080/09537287.2022.2037166,no address|no address|no address,"QUARTZ Laboratory, Supmeca, Saint-Ouen, France",A comparative study of overall equipment effec...,no address


In [106]:
df_ad2.affiliation.value_counts()

affiliation
From Centre Hospitalier Régional Universitaire (CHRU) de Tours, Médecine Intensive Réanimation, INSERM Centre d’Investigation Clinique (CIC) 1415, Clinical Research in Intensive Care and Sepsis–Trial Group for Global Evaluation and Research in Sepsis (CRICS_TRIGGERSep) French Clinical Research Infrastructure Network (F-CRIN) Research Network (S.E., E.M., D.G., C.S.G.), INSERM, Research Center for Respiratory Diseases (S.E., F.B., N.H.-V., R.R.), the University of Tours (S.E., N.H.-V., R.R.), CHRU de Tours, Réanimation Chirurgicale (M.F.), CHRU de Tours, Pharmacie (R.R.), and CHRU de Tours, INSERM CIC 1415 and Université de Tours et Nantes, Methods in Patient-Centered Outcomes and Health Research, INSERM 1246 (E.T.), Tours, Centre Hospitalier et Universitaire (CHU) d’Orléans, Médecine Intensive Réanimation, Orléans (F.B., M.-A.N., D.B.), Médecine Intensive-Réanimation, Hôpitaux Universitaires de Strasbourg, Nouvel Hôpital Civil and INSERM, Unité Mixte de Recherche (UMR) 1260

In [101]:
key ='École doctorale Pratiques et théories du sens'
status_response = requests.get(ROR_API_ENDPOINT + '?' + f'query="{key}"').json()
status_response

{'number_of_results': 0,
 'time_taken': 49,
 'items': [],
 'meta': {'types': [], 'countries': [], 'continents': [], 'statuses': []}}

In [102]:
dict_ror2 = {}
for m, a in enumerate(df_ad2.address.unique()[0:10]):
    row_id = m
    a_split = re.sub(r"[;,\.]", ";", a).split(";")
    nom_univ = re.sub(r"\[.*\]", "", a_split[0]).strip().replace("&egrave;", "è").replace("&", "%26%"). lower()
    #print(nom_univ)
    #key ='École doctorale Pratiques et théories du sens'
    
    try:
        response = requests.get(ROR_API_ENDPOINT + '?' + f'query="{nom_univ}"').json()
        if response["number_of_results"] > 1 :
            for n, x in enumerate(response["items"]):
                if 'education' in x["types"] :
                    datas = response["items"][n]
                    rors = datas["id"]
        elif response["number_of_results"] == 1 :
            datas = response["items"][0]
            rors = datas["id"]
        else:
             rors = "not found"
        dict_ror2[a] = rors
    except:
        print(nom_univ)

    

In [103]:
dict_ror2
#len(df_ad2)

{'Nos últimos trinta anos Nos últimos trinta anos': 'not found',
 'no address': 'not found',
 '« Blacking Up : Une histoire du rock au prisme du Blacking Up : Une histoire du rock au prisme': 'not found',
 '2 . CNRS, UMR 7041 ArScAn -HAROC (Nanterre). UMR 7041 ArScAn -HAROC (Nanterre) CNRS': 'https://ror.org/04hbgcp92',
 'Université Paris 8-LER': 'not found',
 'Actions H 2020 Marie Skłodowska Curie Paris': 'not found',
 'Université de Paris 8 Vincennes - Saint-Denis': 'not found',
 'Zeitschrift für alte Geschichte Zeitschrift für alte Geschichte': 'not found',
 'Analyser le jardin collectif urbain en géographie : une lecture du jardinage par les émotions Analyser le jardin collectif urbain en géographie : une lecture du jardinage par les émotions': 'not found',
 "Érudit est un consortium interuniversitaire sans but lucratif composé de l'Université de Montréal, l'Université Laval et l'Université du Québec à Montréal. Il Érudit est un consortium interuniversitaire sans but lucratif compo

In [105]:
dict_ror2
df_ad2["ror"] = df_ad2.address.map(dict_ror2.get)
df_ad2

,id,struc_address,affiliation,title,address,ror
0,doi10.1590/198053142861,Nos últimos trinta anos Nos últimos trinta anos,Universidade Paris 13; International Associati...,A educação infantil e a questão da escola: o c...,Nos últimos trinta anos Nos últimos trinta anos,not found
1,doi10.1080/15592324.2021.2004769,no address|no address|no address|no address|no...,Laboratoire Interdisciplinaire Des Énergies de...,Our sisters the plants? notes from phylogeneti...,no address,not found
2,doi10.1080/15592324.2021.2004769,no address|no address|no address|no address|no...,Laboratoire Interdisciplinaire Des Énergies de...,Our sisters the plants? notes from phylogeneti...,no address,not found
3,doi10.1080/15592324.2021.2004769,no address|no address|no address|no address|no...,Laboratoire Interdisciplinaire Des Énergies de...,Our sisters the plants? notes from phylogeneti...,no address,not found
4,doi10.1080/15592324.2021.2004769,no address|no address|no address|no address|no...,Laboratoire Interdisciplinaire Des Énergies de...,Our sisters the plants? notes from phylogeneti...,no address,not found
...,...,...,...,...,...,...
57630,doi10.1080/09537287.2022.2037166,no address|no address|no address,"QUARTZ Laboratory, IUT de Montreuil, Montreuil...",A comparative study of overall equipment effec...,no address,not found
57631,doi10.1080/09537287.2022.2037166,no address|no address|no address,"QUARTZ Laboratory, IUT de Montreuil, Montreuil...",A comparative study of overall equipment effec...,no address,not found
57632,doi10.1080/09537287.2022.2037166,no address|no address|no address,"QUARTZ Laboratory, Supmeca, Saint-Ouen, France",A comparative study of overall equipment effec...,no address,not found
57633,doi10.1080/09537287.2022.2037166,no address|no address|no address,"QUARTZ Laboratory, Supmeca, Saint-Ouen, France",A comparative study of overall equipment effec...,no address,not found
